# Gaussian Processes — Exercises

Companion to the note [Gaussian Processes](Gaussian%20Processes.md).

Practise GPs as distributions over functions: prior samples, the regression posterior $\mu_*,\Sigma_*$, Gaussian conditioning, the marginal likelihood and hyperparameter selection, calibrated uncertainty, Bayesian optimisation, and the link to Bayesian linear regression, by hand and from scratch in NumPy.

**18 exercises** · 🧠 Concept ×4 · ✍️ By hand ×7 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.stats import multivariate_normal
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
rng = np.random.default_rng(0)

def rbf(A, B, ell=1.0, sf2=1.0):
    # Squared-exponential kernel for 1-D inputs: sf2 * exp(-(a-b)^2 / (2 ell^2))
    A, B = np.asarray(A, float).reshape(-1), np.asarray(B, float).reshape(-1)
    return sf2 * np.exp(-(A[:, None] - B[None, :]) ** 2 / (2 * ell ** 2))

# Shared regression data: y = sin(x) + noise, noise std 0.1
x_gp = np.sort(rng.uniform(0, 5, 20))
y_gp = np.sin(x_gp) + 0.1 * rng.normal(size=20)
x_star = np.linspace(0, 6, 50)

## Part A · Concepts

### Exercise 1 · A distribution over functions
*🧠 Concept · ★☆☆*

Explain in your own words what $f\sim\mathcal{GP}(m,k)$ means. What do $m$ and $k$ specify, why is 'any finite set of outputs is jointly Gaussian' enough to define it,
and how do you draw a sample function on a computer?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A function evaluated at $n$ input points is just a vector of length $n$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

For any inputs $x_1,\dots,x_n$, the vector $(f(x_1),\dots,f(x_n))$ is $\mathcal N(\mathbf m,K)$ with $\mathbf m_i=m(x_i)$ and $K_{ij}=k(x_i,x_j)$. $m$ is the expected function
(often 0), $k$ says how strongly values at two inputs co-vary, which encodes smoothness, length scale and amplitude. Because Gaussians are consistent under marginalisation, these
finite-dimensional distributions fit together into one process (Kolmogorov). To 'draw a function', choose a fine grid, build $K$ and sample $\mathcal N(\mathbf m,K)$ (Exercise 12).

</details>

### Exercise 2 · Reading the hyperparameters
*🧠 Concept · ★☆☆*

For the kernel $k(x,x')=\sigma_f^2\exp\big(-\frac{(x-x')^2}{2\ell^2}\big)$ plus noise $\sigma^2$: describe prior samples and the posterior fit when
(a) $\ell$ is very small, (b) $\ell$ is very large, (c) $\sigma^2$ is too small, (d) $\sigma^2$ is too large. What does $\sigma_f^2$ control?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\ell$ is the distance over which function values stay correlated.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Wiggly samples; the posterior mean interpolates noise and reverts to the prior mean quickly between points, with wide error bars (overfitting).
(b) Almost straight/constant samples; the fit is too smooth (underfitting). (c) The mean is forced through every noisy point (wiggly, overconfident).
(d) The data are treated as mostly noise; the mean is flat and the uncertainty large. $\sigma_f^2$ is the vertical scale: the prior variance of $f(x)$ far from data.
All of these can be chosen by maximising the marginal likelihood (Exercise 15).

</details>

### Exercise 3 · GP mean = kernel ridge regression
*🧠 Concept · ★★☆*

Compare the GP posterior mean $\mu_*=k_*^\top(K+\sigma^2I)^{-1}y$ with the [[Kernel Methods|kernel ridge]] prediction. What does a GP add, and name two applications where
that extra output is essential.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Set $\lambda=\sigma^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With $\lambda=\sigma^2$ they are **identical**: $\alpha=(K+\lambda I)^{-1}y$, $f(x)=\sum_i\alpha_ik(x_i,x)$. The GP additionally gives the predictive variance $\Sigma_*$ (uncertainty that
grows away from data) and a principled way to set hyperparameters (marginal likelihood instead of cross-validation). Essential for **Bayesian optimisation** (decide where to evaluate next,
Exercise 17), active learning, safety-critical prediction, and surrogate models of expensive simulations.

</details>

### Exercise 4 · The cubic wall
*🧠 Concept · ★★☆*

Where does the $O(N^3)$ cost come from, and what is the memory cost? Explain the idea of **inducing points** (sparse GPs) in a few sentences, and what is lost.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Factorising $K+\sigma^2I$ once is the dominating step.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Training requires a Cholesky factorisation of the $N\times N$ matrix $K+\sigma^2I$ ($O(N^3)$ time, $O(N^2)$ memory); each prediction then costs $O(N)$ for the mean and $O(N^2)$ for the variance.
Sparse GPs summarise the data through $M\ll N$ inducing inputs $Z$: the function values at $Z$ act as a bottleneck, giving a rank-$M$ approximation of $K$ (like Nyström) and cost $O(NM^2)$.
The price is a smoother, less expressive posterior and often underestimated variance far from inducing points; $Z$ is usually optimised with the marginal-likelihood bound.

</details>

## Part B · By hand

### Exercise 5 · Prior correlations
*✍️ By hand · ★☆☆*

With $k(x,x')=\exp(-\frac{(x-x')^2}{2\ell^2})$, what is the correlation between $f(0)$ and $f(2)$ for $\ell=1$ and for $\ell=2$? At what distance does the correlation drop to $0.5$ for $\ell=1$?

In [ ]:
corr_l1 = None
corr_l2 = None
d_half = None

check('l=1', corr_l1, rbf(0, 2, 1.0)[0, 0], tol=1e-4)
check('l=2', corr_l2, rbf(0, 2, 2.0)[0, 0], tol=1e-4)
check('half-correlation distance', d_half, np.sqrt(2*np.log(2)), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

With $\sigma_f^2=1$ the covariance is the correlation. Solve $e^{-d^2/2}=0.5$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\ell=1$: $e^{-2}\approx0.135$. $\ell=2$: $e^{-0.5}\approx0.607$. Half-correlation: $d=\sqrt{2\ln2}\approx1.177$ (in units of $\ell$).

```python
corr_l1 = np.exp(-2)
corr_l2 = np.exp(-0.5)
d_half = np.sqrt(2 * np.log(2))
```

</details>

### Exercise 6 · Posterior from one observation
*✍️ By hand · ★☆☆*

Kernel $\exp(-(x-x')^2/2)$, noise $\sigma^2=0.1$, one observation $x_1=0$, $y_1=1$. Compute $\mu_*$ and $\Sigma_*$ (variance of $f$) at $x_*=1$ and at $x_*=0$.

In [ ]:
mu_at1 = None
var_at1 = None
mu_at0 = None
var_at0 = None

_gp = GaussianProcessRegressor(RBF(1.0), alpha=0.1, optimizer=None).fit([[0.]], [1.])
_m, _s = _gp.predict([[1.], [0.]], return_std=True)
check('mu(1)', mu_at1, _m[0], tol=1e-4); check('var(1)', var_at1, _s[0]**2, tol=1e-4)
check('mu(0)', mu_at0, _m[1], tol=1e-4); check('var(0)', var_at0, _s[1]**2, tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

All matrices are $1\times1$: $K+\sigma^2=1.1$, $k_*=e^{-1/2}\approx0.6065$ at $x_*=1$, $k_{**}=1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

At $x_*=1$: $\mu_*=0.6065/1.1\approx0.551$, $\Sigma_*=1-0.6065^2/1.1\approx0.666$.
At $x_*=0$: $\mu_*=1/1.1\approx0.909$ (shrunk towards the prior mean 0 because of noise), $\Sigma_*=1-1/1.1\approx0.0909$.

```python
mu_at1 = np.exp(-0.5) / 1.1
var_at1 = 1 - np.exp(-1) / 1.1
mu_at0 = 1 / 1.1
var_at0 = 1 - 1 / 1.1
```

</details>

### Exercise 7 · Posterior from two observations
*✍️ By hand · ★★☆*

Same kernel and noise ($\sigma^2=0.1$), observations $X=(0,1)$, $y=(1,2)$. Compute $\mu_*$ and $\Sigma_*$ at $x_*=0.5$ by hand (invert the $2\times2$ matrix).

In [ ]:
mu_two = None
var_two = None

_gp = GaussianProcessRegressor(RBF(1.0), alpha=0.1, optimizer=None).fit([[0.], [1.]], [1., 2.])
_m, _s = _gp.predict([[0.5]], return_std=True)
check('mu(0.5)', mu_two, _m[0], tol=1e-3); check('var(0.5)', var_two, _s[0]**2, tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

$K+\sigma^2I=\begin{pmatrix}1.1&0.6065\\0.6065&1.1\end{pmatrix}$, $k_*=(e^{-1/8},e^{-1/8})\approx(0.8825,0.8825)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Because $k_*$ is proportional to $(1,1)$, an eigenvector of $K+\sigma^2I$ with eigenvalue $1.1+0.6065$: $k_*^\top(K+\sigma^2I)^{-1}=\frac{0.8825}{1.7065}(1,1)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$k_*^\top(K+\sigma^2I)^{-1}=0.5171\,(1,1)$. So $\mu_*=0.5171\cdot(1+2)\approx1.551$ and $\Sigma_*=1-0.5171\cdot2\cdot0.8825\approx0.0873$.
Between two nearby observations the variance is much smaller than with one observation (0.666 in Exercise 6).

```python
_c = np.exp(-1/8) / (1.1 + np.exp(-0.5))
mu_two = _c * 3
var_two = 1 - 2 * _c * np.exp(-1/8)
```

</details>

### Exercise 8 · Deriving the posterior by conditioning
*✍️ By hand · ★★☆*

Write the joint prior of the noisy training targets $y$ and the test values $f_*$ as one Gaussian and use the conditioning formula
$a\mid b\sim\mathcal N\big(\mu_a+\Sigma_{ab}\Sigma_{bb}^{-1}(b-\mu_b),\ \Sigma_{aa}-\Sigma_{ab}\Sigma_{bb}^{-1}\Sigma_{ba}\big)$ to obtain the note's $\mu_*$ and $\Sigma_*$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$y=f(X)+\varepsilon$ with independent noise, so $\operatorname{Cov}(y,y)=K+\sigma^2I$ but $\operatorname{Cov}(f_*,y)=k_*^\top$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\begin{pmatrix}y\\f_*\end{pmatrix}\sim\mathcal N\left(0,\begin{pmatrix}K+\sigma^2I&k_*\\k_*^\top&k_{**}\end{pmatrix}\right)$.
Conditioning $f_*$ on $y$ (zero means): $\mu_*=k_*^\top(K+\sigma^2I)^{-1}y$ and $\Sigma_*=k_{**}-k_*^\top(K+\sigma^2I)^{-1}k_*$. ✔
The posterior variance does not depend on $y$, only on where the inputs are; the predictive distribution of a new noisy $y_*$ adds $\sigma^2$.

</details>

### Exercise 9 · Noise-free interpolation
*✍️ By hand · ★★☆*

Show that with $\sigma^2=0$ (and $K$ invertible) the posterior at a training input $x_i$ has mean $y_i$ and variance $0$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

At $x_*=x_i$, $k_*$ is the $i$-th column of $K$: $k_*=Ke_i$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mu_*=(Ke_i)^\top K^{-1}y=e_i^\top KK^{-1}y=y_i$ and $\Sigma_*=k(x_i,x_i)-e_i^\top KK^{-1}Ke_i=K_{ii}-K_{ii}=0$.
The GP interpolates the data exactly and is certain there. In practice a small 'jitter' ($10^{-8}$–$10^{-6}$) is added for numerical stability, and with real noise $\sigma^2>0$ it no longer interpolates.

</details>

### Exercise 10 · The log marginal likelihood
*✍️ By hand · ★★☆*

The note says hyperparameters are found by maximising the marginal likelihood
$\log p(y\mid X)=-\tfrac12y^\top(K+\sigma^2I)^{-1}y-\tfrac12\log|K+\sigma^2I|-\tfrac N2\log2\pi$.
(a) Interpret the first two terms. (b) Evaluate it for one observation $y_1=1$ with $k(x_1,x_1)=1$ and $\sigma^2=0.1$.

In [ ]:
lml_one = None

check('log marginal likelihood', lml_one, multivariate_normal(mean=[0], cov=[[1.1]]).logpdf([1.]), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$y\sim\mathcal N(0,K+\sigma^2I)$ is just a Gaussian density evaluated at the data.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) The first term is the **data fit** (small if $y$ is likely under the prior covariance); the second is a **complexity penalty** (flexible models spread probability over many datasets,
so $|K+\sigma^2I|$ is large). Their trade-off is an automatic Occam's razor, see [[Bayesian Inference]].
(b) $-\tfrac12\cdot\tfrac1{1.1}-\tfrac12\log1.1-\tfrac12\log2\pi=-0.4545-0.0477-0.9189\approx-1.421$.

```python
lml_one = -0.5 / 1.1 - 0.5 * np.log(1.1) - 0.5 * np.log(2 * np.pi)
```

</details>

### Exercise 11 · Bayesian linear regression is a GP
*✍️ By hand · ★★☆*

Let $f(x)=w^\top\phi(x)$ with prior $w\sim\mathcal N(0,\alpha^{-1}I)$. Show that $f$ is a GP with mean 0 and kernel $k(x,x')=\alpha^{-1}\phi(x)^\top\phi(x')$.
What is the rank of its Gram matrix, and why does that limit the model?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$f$ is a linear function of the Gaussian vector $w$; compute $\mathbb E[f(x)f(x')]$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Any finite vector $(f(x_1),\dots,f(x_n))=\Phi w$ is a linear map of a Gaussian, hence Gaussian, with mean $0$ and covariance $\Phi\,\alpha^{-1}I\,\Phi^\top$, i.e.
$k(x,x')=\alpha^{-1}\phi(x)^\top\phi(x')$. The Gram matrix has rank at most $M$ (number of basis functions): the prior only contains functions in an $M$-dimensional space,
so with many data points the model cannot be more flexible than $M$ parameters ('finite-rank kernel' in the note). The RBF kernel corresponds to infinitely many basis functions.
You will verify the equivalence numerically in Exercise 18.

</details>

## Part C · Code from scratch

### Exercise 12 · Sampling from the prior
*💻 Code · ★☆☆*

Write `sample_prior(x, ell, n_samples, seed)` that draws functions from the zero-mean GP with the setup's `rbf` kernel via a Cholesky factor
($f=Lz$, $z\sim\mathcal N(0,I)$; add jitter $10^{-8}I$). Verify that the empirical covariance of 5000 samples on 10 grid points approximates $K$.
Optional: plot 5 samples for $\ell\in\{0.3,1,3\}$.

In [ ]:
def sample_prior(x, ell=1.0, n_samples=5, seed=0):
    # TODO: return array (n_samples, len(x))
    return None

x10 = np.linspace(0, 3, 10)
F = sample_prior(x10, 1.0, 5000)

check('empirical covariance ≈ K', None if F is None else np.cov(F.T), rbf(x10, x10, 1.0), tol=0.06)

<details>
<summary><b>💡 Hint</b></summary>

`L = np.linalg.cholesky(K + 1e-8 * np.eye(n))`; samples `(L @ z).T` with `z` of shape (n, n_samples).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\operatorname{Cov}(Lz)=LL^\top=K$. The jitter fixes the Cholesky failing on the numerically singular RBF Gram matrix (nearby grid points are almost perfectly correlated).
Small $\ell$ gives wiggly samples, large $\ell$ smooth ones.

```python
def sample_prior(x, ell=1.0, n_samples=5, seed=0):
    r = np.random.default_rng(seed)
    K = rbf(x, x, ell)
    L = np.linalg.cholesky(K + 1e-8 * np.eye(len(x)))
    return (L @ r.normal(size=(len(x), n_samples))).T

x10 = np.linspace(0, 3, 10)
F = sample_prior(x10, 1.0, 5000)
```

</details>

### Exercise 13 · GP regression with Cholesky
*💻 Code · ★★☆*

Implement `gp_posterior(x, y, xs, ell, sf2, noise)` returning the posterior mean and the **variance of $f$** at `xs`, using a Cholesky factorisation
($L=\text{chol}(K+\sigma^2I)$, $\alpha=L^\top\backslash(L\backslash y)$, $v=L\backslash k_*$, $\Sigma_*=k_{**}-v^\top v$). Apply it to `x_gp, y_gp` with $\ell=1$, $\sigma_f^2=1$, $\sigma^2=0.01$
and compare with sklearn.

In [ ]:
def gp_posterior(x, y, xs, ell=1.0, sf2=1.0, noise=0.01):
    # TODO: return (mean, var)
    return None, None

mu_gp, var_gp = gp_posterior(x_gp, y_gp, x_star)

_gp = GaussianProcessRegressor(RBF(1.0), alpha=0.01, optimizer=None).fit(x_gp[:, None], y_gp)
_m, _s = _gp.predict(x_star[:, None], return_std=True)
check('posterior mean', mu_gp, _m, tol=1e-6); check('posterior variance', var_gp, _s**2, tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

`scipy.linalg.solve_triangular(L, b, lower=True)` or simply `np.linalg.solve(L, b)` for the triangular solves. The diagonal of $v^\top v$ is `np.sum(v**2, axis=0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The Cholesky route is the standard, numerically stable implementation (Rasmussen & Williams, Alg. 2.1): one $O(N^3)$ factorisation, then cheap triangular solves.
Note how the variance grows beyond $x=5$, where there is no data, back towards $\sigma_f^2=1$.

```python
def gp_posterior(x, y, xs, ell=1.0, sf2=1.0, noise=0.01):
    L = np.linalg.cholesky(rbf(x, x, ell, sf2) + noise * np.eye(len(x)))
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
    Ks = rbf(x, xs, ell, sf2)
    v = np.linalg.solve(L, Ks)
    return Ks.T @ alpha, sf2 - np.sum(v ** 2, axis=0)

mu_gp, var_gp = gp_posterior(x_gp, y_gp, x_star)
print('variance at x=2.5 and x=6:', var_gp[np.argmin(abs(x_star-2.5))], var_gp[-1])
```

</details>

### Exercise 14 · Computing the marginal likelihood
*💻 Code · ★★☆*

Implement `gp_lml(x, y, ell, sf2, noise)` using the Cholesky factor: $\log|K+\sigma^2I|=2\sum_i\log L_{ii}$. Compare with sklearn's
`log_marginal_likelihood_value_` and with `scipy.stats.multivariate_normal.logpdf`.

In [ ]:
def gp_lml(x, y, ell=1.0, sf2=1.0, noise=0.01):
    # TODO
    return None

lml_val = gp_lml(x_gp, y_gp)

_gp = GaussianProcessRegressor(RBF(1.0), alpha=0.01, optimizer=None).fit(x_gp[:, None], y_gp)
check('LML vs sklearn', lml_val, _gp.log_marginal_likelihood_value_)
check('LML vs scipy', lml_val, multivariate_normal(mean=np.zeros(20), cov=rbf(x_gp, x_gp) + 0.01*np.eye(20)).logpdf(y_gp))

<details>
<summary><b>💡 Hint</b></summary>

$y^\top(K+\sigma^2I)^{-1}y=\lVert L^{-1}y\rVert^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Never compute `np.log(np.linalg.det(...))`: the determinant under- or overflows for moderate $N$, while the sum of log-diagonal entries is stable.

```python
def gp_lml(x, y, ell=1.0, sf2=1.0, noise=0.01):
    L = np.linalg.cholesky(rbf(x, x, ell, sf2) + noise * np.eye(len(x)))
    a = np.linalg.solve(L, y)
    return -0.5 * a @ a - np.sum(np.log(np.diag(L))) - 0.5 * len(x) * np.log(2 * np.pi)

lml_val = gp_lml(x_gp, y_gp)
print(lml_val)
```

</details>

### Exercise 15 · Choosing the length scale
*💻 Code · ★★☆*

Evaluate your `gp_lml` for $\ell$ on `np.logspace(-1, 1, 400)` (with $\sigma_f^2=1$, $\sigma^2=0.01$) and pick the best. Compare with sklearn's optimiser
(`GaussianProcessRegressor(RBF(1.0, (1e-2, 1e2)), alpha=0.01)`, which optimises only $\ell$ here). Store `best_ell`.

In [ ]:
ells = np.logspace(-1, 1, 400)
best_ell = None

_gp = GaussianProcessRegressor(RBF(1.0, (1e-2, 1e2)), alpha=0.01).fit(x_gp[:, None], y_gp)
check('best length scale (within 2%)', best_ell, _gp.kernel_.length_scale, tol=0.02 * _gp.kernel_.length_scale)

<details>
<summary><b>💡 Hint</b></summary>

`ells[np.argmax([gp_lml(x_gp, y_gp, l) for l in ells])]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The LML curve has a clear maximum around $\ell\approx1.8$ for this sine data; very small $\ell$ is punished by the complexity term, very large $\ell$ by the data-fit term.
Gradient-based optimisers (as in sklearn) find the same point; with several hyperparameters, use multiple restarts since the LML can be multimodal.

```python
ells = np.logspace(-1, 1, 400)
lmls = np.array([gp_lml(x_gp, y_gp, l) for l in ells])
best_ell = ells[np.argmax(lmls)]
print('best ell', best_ell)
```

</details>

### Exercise 16 · Are the error bars honest?
*💻 Code · ★★☆*

Draw 1000 test inputs uniformly in $[0,5]$ with targets $\sin(x)+0.1\varepsilon$. Using your `gp_posterior` with `best_ell`, compute the fraction of test targets inside the
95% **predictive** interval $\mu_*\pm1.96\sqrt{\Sigma_*+\sigma^2}$. Also compute the coverage when you (wrongly) leave out $\sigma^2$. Store `coverage` and `coverage_no_noise`.

In [ ]:
coverage = None
coverage_no_noise = None

check('predictive intervals are calibrated (0.92-0.98)', None if coverage is None else 0.92 <= coverage <= 0.98, True)
check('ignoring noise gives over-confident intervals', None if coverage_no_noise is None else coverage_no_noise < coverage - 0.2, True)

<details>
<summary><b>💡 Hint</b></summary>

Use a fresh `np.random.default_rng(7)`. The latent variance $\Sigma_*$ describes $f$; a new $y$ also has noise $\sigma^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With the noise term, about 95% of test targets fall inside the intervals: the model is well calibrated because its assumptions (smooth function, Gaussian noise with the right
variance) hold. Without $\sigma^2$ the bands only cover the uncertainty about $f$, which is tiny inside the data range, so coverage collapses (≈ 0.64 here instead of ≈ 0.95).

```python
r = np.random.default_rng(7)
xt = r.uniform(0, 5, 1000)
yt = np.sin(xt) + 0.1 * r.normal(size=1000)
m, v = gp_posterior(x_gp, y_gp, xt, ell=best_ell)
coverage = np.mean(np.abs(yt - m) <= 1.96 * np.sqrt(v + 0.01))
coverage_no_noise = np.mean(np.abs(yt - m) <= 1.96 * np.sqrt(np.maximum(v, 0)))
print(coverage, coverage_no_noise)
```

</details>

### Exercise 17 · Bayesian optimisation with UCB
*💻 Code · ★★★*

Maximise the 'expensive' function $g(x)=\sin(3x)+0.5x$ on $[0,3]$. Start from evaluations at $x=0,1.5,3$, and for 10 iterations: fit your GP ($\ell=0.5$, $\sigma_f^2=1$, noise $10^{-6}$),
evaluate the upper confidence bound $\mu_*+2\sqrt{\Sigma_*}$ on a 601-point grid, and query its maximiser. Store the best $x$ found in `x_best`.

In [ ]:
g = lambda x: np.sin(3 * x) + 0.5 * x
grid_bo = np.linspace(0, 3, 601)
x_best = None

check('found the global maximum (within 0.02)', x_best, grid_bo[np.argmax(g(grid_bo))], tol=0.02)

<details>
<summary><b>💡 Hint</b></summary>

Keep arrays `Xo, Yo` of evaluated points and append each new query. UCB trades off exploitation (high $\mu$) and exploration (high variance).

</details>

<details>
<summary><b>✅ Solution</b></summary>

After a couple of exploratory queries the method concentrates near $x\approx2.675$, the global maximum ($g\approx2.32$), using 13 evaluations in total instead of a full grid.
This is how GPs are used for hyperparameter tuning ([[Cross-Validation and Model Selection]]); Expected Improvement is a common alternative acquisition function.

```python
g = lambda x: np.sin(3 * x) + 0.5 * x
grid_bo = np.linspace(0, 3, 601)
Xo = np.array([0, 1.5, 3.]); Yo = g(Xo)
for it in range(10):
    m, v = gp_posterior(Xo, Yo, grid_bo, ell=0.5, sf2=1.0, noise=1e-6)
    xn = grid_bo[np.argmax(m + 2 * np.sqrt(np.maximum(v, 0)))]
    Xo, Yo = np.r_[Xo, xn], np.r_[Yo, g(xn)]
x_best = Xo[np.argmax(Yo)]
print('queries:', np.round(Xo, 3), '| best x:', x_best)
```

</details>

### Exercise 18 · Bayesian linear regression = GP, numerically
*💻 Code · ★★☆*

With $\phi(x)=(1,x,x^2)$, prior $w\sim\mathcal N(0,\alpha^{-1}I)$, $\alpha=1$ and noise $\sigma^2=0.1$, compute predictions on `x_star` for the data `x_gp, y_gp` in two ways:
(a) the BLR posterior mean $\phi(x_*)^\top m_N$ with $m_N=\sigma^{-2}S_N\Phi^\top y$, $S_N=(\alpha I+\sigma^{-2}\Phi^\top\Phi)^{-1}$; (b) the GP mean with kernel $k=\alpha^{-1}\phi(x)^\top\phi(x')$.
Also compare the predictive variances $\phi_*^\top S_N\phi_*$ vs $\Sigma_*$.

In [ ]:
phi = lambda x: np.c_[np.ones(len(x)), x, x ** 2]
mean_blr = None
mean_gp_lin = None
var_blr = None
var_gp_lin = None

check('means agree', mean_blr, mean_gp_lin if mean_gp_lin is not None else np.nan, tol=1e-6)
check('variances agree', var_blr, var_gp_lin if var_gp_lin is not None else np.nan, tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

GP: `K = phi(x) @ phi(x).T / alpha`, `ks = phi(x) @ phi(xs).T / alpha`; then the usual formulas with $K+\sigma^2I$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Both give identical means and variances (to round-off): weight-space ($3\times3$ inverse) and function-space ($20\times20$ inverse) views of the same model.
The weight-space view is cheaper here ($M=3\ll N$); the function-space view is the only option for infinite-dimensional kernels like the RBF.

```python
phi = lambda x: np.c_[np.ones(len(x)), x, x ** 2]
alpha_p, s2 = 1.0, 0.1
P, Ps = phi(x_gp), phi(x_star)
S_N = np.linalg.inv(alpha_p * np.eye(3) + P.T @ P / s2)
m_N = S_N @ P.T @ y_gp / s2
mean_blr = Ps @ m_N
var_blr = np.sum(Ps @ S_N * Ps, axis=1)
K = P @ P.T / alpha_p; ks = P @ Ps.T / alpha_p; kss = np.sum(Ps * Ps, axis=1) / alpha_p
A = np.linalg.solve(K + s2 * np.eye(len(x_gp)), ks)
mean_gp_lin = A.T @ y_gp
var_gp_lin = kss - np.sum(ks * A, axis=0)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Gaussian Processes](Gaussian%20Processes.md).*